# GeoXMate Modeling Pipeline

This notebook demonstrates a complete 3D geological modeling workflow:

- Structural grid construction
- Horizon interpolation (ZMAP)
- Facies upscaling
- Sequential Indicator Simulation (SIS)
- Seismic mapping
- Grid quality control (cell angle)
- Visualization

Author: Khaled Saleh

In [1]:
import sys
import os

sys.path.append("../src")
sys.path.append("C:\\Users\\DELL\\Desktop\\PyGeoModeler\\src")

import numpy as np
import pandas as pd

from grid.geogrid import GeoGrid
from data_io.zmap_reader import read_zmap, interpolate_zmap_to_grid_xy
from modeling.facies import upscale_facies
from modeling.simulation import run_sis_simulation
from modeling.seismic import map_seismic_to_grid

from qc.cell_angle import compute_cell_angle
from visualization.pyvista_viewer import show_grid, facies_colormap


### 1. Build Grid

In [3]:
grid = GeoGrid()

grid.set_parameters({
    "X_min": 0,
    "X_max": 1000,
    "Y_min": 0,
    "Y_max": 1000,
    "Z_min": -100,
    "Z_max": 0,
    "dx": 50,
    "dy": 50,
    "cell_thickness": 2,
    "zones_config": {
        "above_ARG": None,
        "ARG_UB": None,
        "UB_LB": None,
        "LB_KH": None,
        "below_KH": None
    },
    "layers_config": {
        "above_ARG": 2,
        "ARG_UB": 2,
        "UB_LB": 2,
        "LB_KH": 2,
        "below_KH": 2
    }
})

grid.build_xy_grid()

(array([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0],
        [ 50,  50,  50,  50,  50,  50,  50,  50,  50,  50,  50,  50,  50,
          50,  50,  50,  50,  50,  50,  50],
        [100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
         100, 100, 100, 100, 100, 100, 100],
        [150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150,
         150, 150, 150, 150, 150, 150, 150],
        [200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200,
         200, 200, 200, 200, 200, 200, 200],
        [250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250,
         250, 250, 250, 250, 250, 250, 250],
        [300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300,
         300, 300, 300, 300, 300, 300, 300],
        [350, 350, 350, 350, 350, 350, 350, 350, 350, 350, 350, 350, 350,
         350, 350, 350, 350, 350, 350, 350],
        [400, 400, 400, 400, 400, 400, 400, 400,

### 🧪 2. Horizons

In [4]:
XX, YY = grid.XX, grid.YY

Z_maps = {
    "ARG": np.full_like(XX, -20),
    "UB": np.full_like(XX, -40),
    "LB": np.full_like(XX, -60),
    "KH": np.full_like(XX, -80),
}

grid.build_3d_grid(Z_maps)

StructuredGrid,Information
N Cells,3610
N Points,4400
X Bounds,"0.000e+00, 9.500e+02"
Y Bounds,"0.000e+00, 9.500e+02"
Z Bounds,"-1.000e+02, 0.000e+00"
Dimensions,"20, 20, 11"
N Arrays,0


### 🧪 3. Visual Check

In [5]:
show_grid(grid.grid, show_edges=True)

Widget(value='<iframe src="http://localhost:53311/index.html?ui=P_0x1d333455d30_0&reconnect=auto" class="pyvis…

### 🧪 4. Create Dummy Wells

In [6]:
df = pd.DataFrame({
    "Easting": np.random.uniform(0, 1000, 200),
    "Northing": np.random.uniform(0, 1000, 200),
    "TVDSS": np.random.uniform(-100, 0, 200),
    "FACIES": np.random.randint(0, 4, 200)
})

df.to_csv("test_wells.csv", index=False)

In [7]:
facies = upscale_facies(grid, "test_wells.csv")
grid.add_property("FACIES", facies)

In [8]:
show_grid(grid.grid, scalar="FACIES", cmap=facies_colormap())

Widget(value='<iframe src="http://localhost:53311/index.html?ui=P_0x1d3385d4a50_1&reconnect=auto" class="pyvis…

In [9]:
facies_sim = run_sis_simulation(grid, n_neighbors=5)
grid.add_property("FACIES_SIS", facies_sim)

SIS (layer 9): 100%|████████████████████████████████████████████████████████████| 361/361 [00:00<00:00, 12148.62cell/s]


In [10]:
show_grid(grid.grid, scalar="FACIES_SIS", cmap=facies_colormap())

Widget(value='<iframe src="http://localhost:53311/index.html?ui=P_0x1d3385d7110_2&reconnect=auto" class="pyvis…

In [12]:
layer_ids = grid.get_layer_ids().flatten(order='F')

k = 3
mask = layer_ids == k

subgrid = grid.grid.extract_cells(np.where(mask)[0])

show_grid(subgrid, scalar="FACIES_SIS", cmap=facies_colormap())

Widget(value='<iframe src="http://localhost:53311/index.html?ui=P_0x1d3386f9a90_4&reconnect=auto" class="pyvis…